In [ ]:
%load_ext autoreload
%autoreload 2

## Load Raw

In [ ]:
from smlm.io import load_raw
from smlm import viz

# EMCCD 的 SIF —— em_gain 自动从头信息读出
src = r"E:\CodingProj\Data\Sample_Data\20260703_Cy3\532nm_1p5x_0p1S_500frames_1p1mW_cy3_fov1.sif"
raw = load_raw(src)
print(raw.data.shape, raw.n_frames, raw.em_gain, raw.exposure_time)
pixel_size=106.6 # nm/pixel
# sCMOS 的 TIF,每 5 帧取 1 帧
#raw = load_raw("cell01.tif", interval=5)
#print(raw.images.shape, raw.em_gain)   # em_gain 为 None

In [ ]:
# 接下来交给 camera 层转光子
from smlm.camera import EMCCD
cam = EMCCD(conv_factor=5.75, em_gain=raw.em_gain)
image = cam.process(raw, pixel_size=106.6)

## Preprocessing

In [ ]:
from smlm.imgprocess import Pipeline, Crop, SelectFrames
# 预处理: 裁剪 ROI + 选前 2000 帧
prep = Pipeline(Crop(y=(100, 400), x=(100, 400)),
                SelectFrames(stop=500))
stack = prep(image)

## Detect

In [ ]:
from smlm.localize import LocalMaxDetector
import numpy as np

detector=LocalMaxDetector(k_sigma=4.0, edge=5, f_em=np.sqrt(2),
                 bg_window=15, max_size=7, min_neighbors=2, verbose=True)
cand = detector.detect(stack)

In [ ]:
ax = viz.overlay_points(stack, cand, frames=0, color='r', marker='o')

## Localize

In [ ]:
from smlm.localize import MLELocalizer
localizer = MLELocalizer(edge=5, F_EM=np.sqrt(2.0), use_pixel=False,
                 use_gradient=True, n_jobs=8, verbose=True)
locs = localizer.localize(stack,cand)

In [ ]:
from smlm.localize import filter_locs
from smlm.viz import plot_distributions
locs = filter_locs(locs)
plot_distributions(locs)

## Detect+Localize

In [ ]:
from smlm.localize import ClassicalPipeline, LocalMaxDetector, MLELocalizer, filter_locs
from smlm.io import run_cached
import numpy as np
method = ClassicalPipeline(
    detector=LocalMaxDetector(k_sigma=3.8, edge=5, f_em=np.sqrt(2),
                 bg_window=15, max_size=7, min_neighbors=2, verbose=True),
    localizer = MLELocalizer(edge=5, F_EM=np.sqrt(2.0), use_pixel=False,
                 use_gradient=True, n_jobs=1, verbose=True),
)
locs = run_cached(method, stack, src)   # 首次计算并保存, 之后直接读盘

In [ ]:
from smlm.viz import plot_distributions
locs = filter_locs(locs)
plot_distributions(locs)

In [ ]:
ax = viz.overlay_points(stack, locs, frames=0, color='r', marker='x', size=30, linewidth = 0.5, unit='pixel',pixel_size=pixel_size)

## Drift Correction

In [ ]:
from smlm.drift import CloudDriftEstimator
from smlm.imgprocess import DriftCorrector     # 校正侧已移至 imgprocess

# 1) 估计(只需要定位)
uncertainty = np.median(locs.lp)
est = CloudDriftEstimator(frames_per_seg=10, uncertainty=uncertainty, do_stage2=True)
drift_trace = est.estimate(locs, n_frames=image.n_frames)
#drift_trace = DriftTrace.zeros(n_frames = stack.n_frames, pixel_size=pixel_size)  # 测试零漂移占位

# 2) 校正图像 + 定位(带缓冲裁切, 与 MATLAB correctDrift 对齐)
corr = DriftCorrector(crop=(257, 257), edge=5)
corr_image, corr_locs, keep_idx = corr.apply(trace=drift_trace, image=stack, locs=locs)

In [ ]:
from smlm.viz import plot_drift
fig, ax = plot_drift(drift_trace, ex_time=raw.exposure_time,unit='nm',pixel_size=pixel_size) 


In [ ]:
ax = viz.overlay_points(corr_image, corr_locs, frames=np.arange(0,1), color='r', marker='x', size=30, linewidth = 0.5)

## Emitter Analysis

In [ ]:
from smlm.emitter import analyze_emitters

emitters, emitters_raw = analyze_emitters(
    corr_locs, n_frames=None,
    search_radius=2,
    max_gap=100,
    min_frames=2,
    ecc_th=3.5,
    motion='static',do_merge=False
)

In [ ]:
from smlm.emitter import collect_statistics
stats = collect_statistics(emitters, locs, raw.exposure_time, 0)


In [ ]:
from smlm.viz import plot_trajectories, plot_bleaching_summary, plot_emitter_stats
plot_trajectories(corr_image.data, emitters, linewidth=0.7, frame_idx=0, dpi=250)
plot_emitter_stats(stats)

## Trace Extraction

In [ ]:
from smlm.trace import ApertureTraceExtractor
extractor = ApertureTraceExtractor(edge=3, do_bg_subtract=True, bg_edge=6, excl_radius=4, sigma_clip_n=3)


In [ ]:
traces = extractor.extract(emitters, corr_image.data, corr_image.n_frames)

In [ ]:
from smlm.viz import check_trace
for i in range(10,11):
    # photons_i 长度 == emitters[i].frame.size(每个 localization 一个值)
    photons_i = corr_locs.photons[emitters[i].loc_idx]     # 按你的 Localizations 取法
    check_trace(traces[i], emitter=emitters[i], ex_time=raw.exposure_time,
                mle_brightness=photons_i, use_time=True)

## Trace Analysis

In [ ]:
from smlm.trace import get_analyzer, analyze_traces, plot_state_gallery
specs = get_analyzer("changepoint").param_specs()
for name, (default, help_) in specs.items():
    print(f"{name:18s} = {default!r:12}  {help_}")

In [ ]:
results = analyze_traces(traces, method="changepoint",penalty=3, bleach_tail = 50)

In [ ]:
fig = plot_state_gallery(results, n_max=20)

In [ ]:
from smlm.trace import plot_summary
fig = plot_summary(results)